# Agentic AI Systems — Worklet 15
# Executive Decision Support System
## Lab 1 · Agent vs Chatbot — the Decision-Brief Planner Loop

**Phase A · Foundations** &nbsp;|&nbsp; **Goal:** establish the agent loop (clarify → plan → revise) instead of a one-shot chatbot answer.

---

### The worklet (Chapter 15)

Leadership reviews academic, research, placement and finance performance from separate departmental reports that arrive on different schedules and formats — with no single real-time view of institution-wide KPIs, and no way to drill from a KPI back to its source. The Executive Decision Support System aggregates the four domains, synthesises institution-wide KPIs against NIRF / NAAC / strategic-plan targets, and produces a **decision brief with options and risks**, with drill-down evidence back to the source department.

* **Human authority:** the Principal / Registrar / Governing Council. Agents draft and flag; the named human approves.
* **Primary output:** a decision brief with options and risks.

### What Lab 1 contributes to it

| | |
|---|---|
| **Inputs** | Free-text request from leadership; core parameters. |
| **Outputs** | A clarified request + an editable outline plan for the decision brief. |
| **Agents / nodes** | apniLeap AURA Coordinator — **loop only**, no specialist agents yet. |
| **Deliverable** | A notebook agent that clarifies, plans and revises a decision-brief outline. |
| **Acceptance** | Given a vague request it asks **at least two** clarifying questions and yields a coherent, **editable** plan. |
| **Pitfall to avoid** | Don't answer in one shot — without clarify/revise it is a chatbot, not an agent. |

### The loop this notebook implements

```
Goal ──▶ Plan ──▶ Act ──▶ Observe ──▶ Revise
 │         │        │        │           │
 │         │        │        │           └─ Cell 7  typed patch ops mutate the live plan
 │         │        │        └───────────── Cell 8  the tool's answer becomes plan constraints
 │         │        └────────────────────── Cell 5  read_data_scope_register(), return value checked
 │         └─────────────────────────────── Cell 8  next_step() chooses from goal state alone
 └───────────────────────────────────────── Cell 3  goal as data + the stopping condition
```

### How to run

The notebook runs **with or without a Gemini API key**.

* **With a key** — add `GEMINI_API_KEY` to Colab Secrets (🔑 in the left sidebar) or export it as an environment variable. Every reasoning step then goes to Gemini at temperature 0, with `response_schema` forcing structured JSON.
* **Without a key** — a rule-based `OfflineBrain` returns the same JSON structures, so every cell still executes. The control structure is what this lab grades, and it is identical either way.

**Cell 2 cannot fail.** A missing key, a retired model id, a denied project, an exhausted quota or a mid-run outage all degrade to the offline brain with a printed explanation, rather than raising. A degraded run is still a complete, correct, fully-graded run — only the phrasing changes. Set `FORCE_OFFLINE = True` in Cell 2 to skip Gemini deliberately.

Run top to bottom. Cell 13 executes the rubric as assertions.

---

### Cell 1 · Environment

In [1]:
# =============================================================================
# Cell 1  ·  Environment setup
# =============================================================================
# Installs the Gemini SDK. Safe to re-run; skipped automatically if present.
try:
    import google.genai  # noqa: F401
    print("google-genai already available.")
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "google-genai"], check=False)
    print("google-genai installed.")


google-genai already available.


### Cell 2 · One LLM interface, two backends

Everything the agent needs from a model goes through a single call: `llm.json(task=..., system=..., prompt=..., schema=...) -> dict`.

Two design decisions worth noticing, because both are graded indirectly:

1. **Structured output, not prose.** The Gemini path sets `response_mime_type="application/json"` plus a `response_schema`, so the model is constrained to return exactly the shape the agent code expects. A malformed reply fails schema validation instead of quietly corrupting the plan.
2. **Automatic function calling is disabled.** The lab grades the control structure, so the loop is driven explicitly in Python rather than hidden inside the SDK.

Every call is appended to `llm.calls` — that list *is* the prompt log the deliverable asks for.

In [2]:
# =============================================================================
# Cell 2  ·  LLM backend, with a deterministic fallback that CANNOT fail
# =============================================================================
# The agent talks to exactly one interface: LLM.json(...) -> dict.
#
#   * Gemini when the key works  -> temperature 0, response_mime_type=
#     "application/json" + a response_schema, so the model is *forced* to return
#     the exact structure the agent code expects.
#   * OfflineBrain otherwise     -> a rule-based stand-in returning the same
#     structures.
#
# THIS CELL NEVER RAISES, AND NEITHER DOES ANY CALL BELOW IT.
# A missing key, a retired model id, a denied project, a rate limit or an outage
# all degrade to the offline brain with a printed explanation. The control
# structure is what these labs grade, and it is identical on both paths, so a
# degraded run is still a complete, correct, fully-graded run.
#
# Three layers of protection:
#   1. FORCE_OFFLINE       — skip Gemini entirely if you want a guaranteed run.
#   2. Preflight probe     — one tiny call at startup finds the first model id
#                            this key can actually reach, so a bad id surfaces
#                            HERE with a clear message, not 200 lines later.
#   3. Degrade-on-failure  — if Gemini breaks mid-run, that call and every call
#                            after it transparently uses the offline brain.

import os, re, json, time, textwrap
from typing import Any, Dict, List, Optional

# Set to True to bypass Gemini completely and run purely offline.
FORCE_OFFLINE = False

MODEL_CANDIDATES = [
    "gemini-3.6-flash",     # current default for new API keys
    "gemini-flash-latest",  # rolling alias for the newest flash
    "gemini-2.5-flash",     # older keys may still have access
    "gemini-2.0-flash",
]


def _load_api_key() -> Optional[str]:
    """Colab secret -> environment variable -> None (offline mode)."""
    try:
        from google.colab import userdata  # type: ignore
        key = userdata.get("GEMINI_API_KEY")
        if key:
            return key
    except Exception:
        pass
    return os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")


class GeminiBackend:
    """Deterministic, schema-constrained wrapper with model fall-forward."""

    name = "gemini"
    # Any of these means "this model id is not usable on this key" -> try the
    # next candidate. 403/PERMISSION_DENIED belongs here as much as 404 does:
    # both mean the call will never succeed, so retrying is pointless.
    _MODEL_ERROR_MARKERS = ("NOT_FOUND", "404", "403", "PERMISSION_DENIED",
                            "no longer available", "is not found", "not supported",
                            "denied access", "API_KEY_INVALID", "INVALID_ARGUMENT",
                            "quota", "RESOURCE_EXHAUSTED", "429")

    def __init__(self, api_key: str, candidates: List[str] = None):
        from google import genai
        from google.genai import types
        self._types = types
        self.client = genai.Client(api_key=api_key)
        self.candidates = list(candidates or MODEL_CANDIDATES)
        self.model = self.candidates[0]
        self.model_switches: List[str] = []
        self.probe_errors: List[tuple] = []

    # ---- introspection --------------------------------------------------
    def available_models(self) -> List[str]:
        try:
            return sorted(
                m.name.replace("models/", "") for m in self.client.models.list()
                if "generateContent" in (getattr(m, "supported_actions", None) or []))
        except Exception as exc:
            return [f"(could not list models: {type(exc).__name__}: {exc})"]

    # ---- robust JSON extraction -----------------------------------------
    def _extract_json(self, resp) -> Dict[str, Any]:
        """Newer models emit reasoning parts; take only the real text parts."""
        parsed = getattr(resp, "parsed", None)
        if isinstance(parsed, dict) and parsed:
            return parsed
        chunks = []
        for cand in (getattr(resp, "candidates", None) or []):
            content = getattr(cand, "content", None)
            for part in (getattr(content, "parts", None) or []):
                if getattr(part, "thought", False):      # skip reasoning parts
                    continue
                if getattr(part, "text", None):
                    chunks.append(part.text)
        text = ("".join(chunks) or getattr(resp, "text", "") or "").strip()
        if text.startswith("```"):                        # strip a stray fence
            text = re.sub(r"^```[a-zA-Z]*\s*", "", text)
            text = re.sub(r"\s*```$", "", text)
        if not text:
            raise ValueError("model returned no text content")
        return json.loads(text)

    def _config(self, system: str, schema: Dict[str, Any]):
        types = self._types
        return types.GenerateContentConfig(
            system_instruction=system,
            temperature=0.0,                     # deterministic -> reproducible marks
            seed=7,
            response_mime_type="application/json",
            response_schema=schema,
            # We drive the loop ourselves; automatic function calling would hide
            # the control flow these labs are graded on.
            automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
        )

    # ---- preflight: find a model this key can actually call --------------
    def preflight(self) -> Optional[str]:
        probe_schema = {"type": "object",
                        "properties": {"ok": {"type": "boolean"}},
                        "required": ["ok"]}
        while self.candidates:
            candidate = self.candidates[0]
            try:
                resp = self.client.models.generate_content(
                    model=candidate, contents='Reply with exactly {"ok": true}',
                    config=self._config("Reply only with JSON.", probe_schema))
                self._extract_json(resp)
                self.model = candidate
                return candidate
            except Exception as exc:
                self.probe_errors.append((candidate, f"{type(exc).__name__}: "
                                                     f"{str(exc)[:220]}"))
                self.candidates.pop(0)
        return None

    # ---- normal calls ----------------------------------------------------
    def json(self, system: str, prompt: str, schema: Dict[str, Any],
             retries: int = 3) -> Dict[str, Any]:
        cfg = self._config(system, schema)
        last_err, attempts = None, 0
        while attempts < retries:
            try:
                resp = self.client.models.generate_content(
                    model=self.model, contents=prompt, config=cfg)
                return self._extract_json(resp)
            except Exception as exc:
                last_err = exc
                fatal_for_model = any(m in str(exc) for m in self._MODEL_ERROR_MARKERS)
                if fatal_for_model and len(self.candidates) > 1:
                    retired = self.candidates.pop(0)
                    self.model = self.candidates[0]
                    self.model_switches.append(f"{retired} -> {self.model}")
                    print(f"  [model] '{retired}' unusable on this key; "
                          f"switching to '{self.model}'")
                    continue                     # a switch does not burn a retry
                if fatal_for_model:
                    break                        # no candidates left; stop retrying
                attempts += 1
                if attempts < retries:
                    time.sleep(1.5 * attempts)   # transient 503 etc.
        raise RuntimeError(f"model '{self.model}': {last_err}")


class OfflineBrain:
    """Rule-based stand-in for the LLM.

    Mirrors the reference-implementation style used throughout the lab guide:
    the *control structure* is what these labs grade, so it must remain runnable
    and inspectable with no API key. Handlers are registered by task name.
    """

    name = "offline"

    def __init__(self):
        self.handlers = {}

    def register(self, task: str):
        def deco(fn):
            self.handlers[task] = fn
            return fn
        return deco

    def json(self, system: str, prompt: str, schema: Dict[str, Any],
             task: str = "", payload: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
        if task not in self.handlers:
            raise RuntimeError(f"OfflineBrain has no handler for task '{task}'")
        return self.handlers[task](payload or {})


def _explain_gemini_failure(errors: List[tuple]) -> None:
    print("  Gemini could not be used. What each model id returned:")
    for model, err in errors:
        print(f"    - {model:<22} {err}")
    blob = " ".join(e for _, e in errors)
    print("\n  Most likely cause, based on those messages:")
    if "PERMISSION_DENIED" in blob or "denied access" in blob:
        print("    403 PERMISSION_DENIED — the key authenticated, but the Google Cloud")
        print("    project behind it is not allowed to call the Generative Language API.")
        print("    Usual fixes, in order of how often they work:")
        print("      1. In AI Studio (aistudio.google.com/apikey), delete this key and")
        print("         choose 'Create API key in a NEW project'. A key inherited from an")
        print("         existing/organisation project is the common cause.")
        print("      2. If you are on a college/organisation Google account, use a personal")
        print("         @gmail.com account instead — org policy often blocks this API.")
        print("      3. A brand-new key can take a few minutes to activate; retry shortly.")
    elif "API_KEY_INVALID" in blob or "INVALID_ARGUMENT" in blob:
        print("    The key itself was rejected. Check for a stray space when pasting it")
        print("    into the Colab secret, and that the secret is named GEMINI_API_KEY.")
    elif "RESOURCE_EXHAUSTED" in blob or "429" in blob or "quota" in blob:
        print("    Free-tier quota is exhausted for now. It resets; try again later.")
    else:
        print("    Every candidate model id was rejected. Run llm.available_models()")
        print("    and put a working id at the front of MODEL_CANDIDATES.")
    print("\n  THIS DOES NOT BLOCK THE LAB. The notebook has switched to the offline")
    print("  brain and every cell below runs normally — same loop, same tools, same")
    print("  checks, same marks. Only the phrasing is rule-based instead of generated.")


class LLM:
    """Router: Gemini when it works, OfflineBrain otherwise. Never raises."""

    def __init__(self):
        self.offline = OfflineBrain()
        self.backend = None
        self.degraded_reason: Optional[str] = None
        self.calls: List[Dict[str, Any]] = []      # the prompt log (a deliverable)

        if FORCE_OFFLINE:
            self.degraded_reason = "FORCE_OFFLINE is set to True in this cell"
            return
        key = _load_api_key()
        if not key:
            self.degraded_reason = "no GEMINI_API_KEY found in Colab secrets or environment"
            return
        try:
            candidate_backend = GeminiBackend(key)
            print("  Probing which Gemini model this key can reach...")
            if candidate_backend.preflight():
                self.backend = candidate_backend
            else:
                self.degraded_reason = "no candidate model was callable with this key"
                _explain_gemini_failure(candidate_backend.probe_errors)
        except Exception as exc:
            self.degraded_reason = f"{type(exc).__name__}: {exc}"
            _explain_gemini_failure([("client init", str(exc)[:220])])

    @property
    def mode(self) -> str:
        return "gemini" if self.backend else "offline"

    @property
    def model_label(self) -> str:
        return self.backend.model if self.backend else "rule-based OfflineBrain"

    def available_models(self) -> List[str]:
        if not self.backend:
            return ["(offline mode — no working Gemini backend)"]
        return self.backend.available_models()

    def json(self, *, task: str, system: str, prompt: str,
             schema: Dict[str, Any], payload: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
        source = "offline"
        if self.backend:
            try:
                out = self.backend.json(system, prompt, schema)
                source = f"gemini:{self.backend.model}"
            except Exception as exc:
                # Degrade permanently rather than raise: one API problem must not
                # take down a notebook that is being graded.
                self.degraded_reason = str(exc)
                self.backend = None
                print(f"\n  [degraded] Gemini failed on task '{task}': {exc}")
                print("  [degraded] Switching to the offline brain for this and all")
                print("  [degraded] later calls. Every cell below still runs and still")
                print("  [degraded] passes its checks.\n")
                out = self.offline.json(system, prompt, schema, task=task, payload=payload)
        else:
            out = self.offline.json(system, prompt, schema, task=task, payload=payload)
        self.calls.append({
            "call": len(self.calls) + 1, "task": task, "backend": source,
            "system": system.strip(), "prompt": prompt.strip(), "response": out,
        })
        return out


llm = LLM()
print(f"\nLLM backend  : {llm.mode}")
print(f"Model        : {llm.model_label}")
if llm.mode == "gemini":
    print("Verified     : a live probe call succeeded on this model id.")
    print(f"Fallback order remaining: {llm.backend.candidates}")
else:
    print(f"Reason       : {llm.degraded_reason}")
    print("Impact       : none on grading — the loop, the tools, the logs and every")
    print("               acceptance check below are backend-independent.")
print("Both backends return the same JSON structures, so every cell below runs either way.")



LLM backend  : offline
Model        : rule-based OfflineBrain
Reason       : no GEMINI_API_KEY found in Colab secrets or environment
Impact       : none on grading — the loop, the tools, the logs and every
               acceptance check below are backend-independent.
Both backends return the same JSON structures, so every cell below runs either way.


### Cell 3 · The goal, modelled as data

A goal written as a paragraph cannot be checked. This one is six slots plus a boolean, so `satisfied()` is a real predicate and the stopping condition is **printed before the loop ever runs** — the agent can never decide it is finished for a reason we did not declare up front.

In [3]:
# =============================================================================
# Cell 3  ·  The goal, modelled as data  (+ the stopping condition)
# =============================================================================
# Concept 1: a goal is a *checkable predicate over state*, not a paragraph of
# prose. Six slots must be filled before a plan can be written; `plan_locked`
# is the single stopping condition, and it is declared here — before the loop
# runs — so the agent can never "decide it is done" for a reason we did not
# state up front.

from dataclasses import dataclass, field, asdict

# The four domains this worklet is allowed to reason about (Chapter 15 roster).
VALID_DOMAINS = ["Academic", "Research", "Placement", "Finance"]

# question text used when a slot is still empty. Keyed by slot name.
SLOT_QUESTIONS = {
    "domains":   "Which performance domains must the brief cover — Academic, Research, Placement, Finance, or a subset?",
    "period":    "Which reporting period does this cover (e.g. academic year 2025-26, or a specific quarter)?",
    "kpis":      "Which KPIs should be reported, or should the approved institution-wide KPI set be used in full?",
    "targets":   "Which target set should performance be compared against — NIRF, NAAC, the internal strategic plan, or all three?",
    "drilldown": "Is department-level drill-down required for every KPI, or only for KPIs that miss their target?",
    "output":    "What should leadership receive — a decision brief with options and risks, a dashboard, or both?",
}
REQUIRED_SLOTS = list(SLOT_QUESTIONS)


@dataclass
class Goal:
    """Run-state for the coordinator. Empty slots are what drives the loop."""
    domains:   Optional[List[str]] = None
    period:    Optional[str] = None
    kpis:      Optional[List[str]] = None
    targets:   Optional[List[str]] = None
    drilldown: Optional[str] = None
    output:    Optional[str] = None
    # --- stopping condition ---------------------------------------------
    plan_locked: bool = False
    # --- observations recorded by tool calls (Act -> Observe) ------------
    scope_checked: bool = False
    conflicts: List[Dict[str, Any]] = field(default_factory=list)

    def missing(self) -> List[str]:
        """Only the slots that are actually empty — not a fixed questionnaire."""
        return [s for s in REQUIRED_SLOTS if getattr(self, s) in (None, [], "")]

    def requirements_complete(self) -> bool:
        return not self.missing()

    def satisfied(self) -> bool:
        """THE stopping condition, declared before the loop runs."""
        return self.plan_locked is True

    def snapshot(self) -> Dict[str, Any]:
        return asdict(self)


print("Stopping condition (declared before the loop runs):")
print("    goal.plan_locked is True")
print()
print("Sub-conditions that must hold before the plan may be locked:")
print("    1. goal.missing() == []        -> every required slot is filled")
print("    2. goal.scope_checked is True  -> the data-scope tool has actually been called")
print()
print(f"Required slots ({len(REQUIRED_SLOTS)}): {', '.join(REQUIRED_SLOTS)}")


Stopping condition (declared before the loop runs):
    goal.plan_locked is True

Sub-conditions that must hold before the plan may be locked:
    1. goal.missing() == []        -> every required slot is filled
    2. goal.scope_checked is True  -> the data-scope tool has actually been called

Required slots (6): domains, period, kpis, targets, drilldown, output


### Cell 4 · `clarify()` — ask only what is missing

One call does two jobs: **extract** every slot the request already answers, and **ask** about the ones it does not. Because extraction is data, the loop never re-asks a question that has been answered — and a request that specifies everything gets zero questions (proved in Cell 11).

Note the guardrail after the model returns: the coordinator drops any slot the model tried to fill that was not actually empty, rejects domains outside the approved roster, and re-derives the question list from state. **The model proposes; Python decides.**

In [4]:
# =============================================================================
# Cell 4  ·  clarify()  —  ask ONLY what is missing, never invent
# =============================================================================
# Two jobs in one call:
#   1. EXTRACT every slot the user already supplied (so we never re-ask), and
#   2. ASK a question for each slot that is still empty.
# The response schema forces both halves to come back as data, which is what
# lets the loop below be driven by state instead of by string matching on prose.

import re

CLARIFY_SCHEMA = {
    "type": "object",
    "properties": {
        "status":    {"type": "string", "enum": ["NEEDS_INFO", "REQUIREMENTS_COMPLETE"]},
        "extracted": {
            "type": "object",
            "properties": {
                "domains":   {"type": "array", "items": {"type": "string"}},
                "period":    {"type": "string"},
                "kpis":      {"type": "array", "items": {"type": "string"}},
                "targets":   {"type": "array", "items": {"type": "string"}},
                "drilldown": {"type": "string"},
                "output":    {"type": "string"},
            },
        },
        "questions": {"type": "array", "items": {"type": "string"}},
        "rationale": {"type": "string"},
    },
    "required": ["status", "extracted", "questions", "rationale"],
}

CLARIFY_SYSTEM = """You are the apniLeap AURA Coordinator for Worklet 15, an Executive
Decision Support System serving the Principal / Registrar / Governing Council of an
educational institution. You are at Lab 1: you clarify, plan and revise. You never
produce the decision brief itself and you never state an institutional figure."""

CLARIFY_PROMPT = """A request has arrived from leadership.

REQUEST:
{request}

SLOTS ALREADY FILLED (do not ask about these again):
{filled}

SLOTS STILL EMPTY — ask about exactly these, and nothing else:
{missing}

For each empty slot, the canonical question is:
{question_bank}

Do two things and return them as JSON:

1. "extracted": for every EMPTY slot, if the request text genuinely supplies the
   answer, put the value there. Domains must be drawn only from
   [Academic, Research, Placement, Finance]. Leave a slot out entirely if the
   request does not state it — never guess, never fill it with a default.
2. "questions": one clear question for every slot that is still empty after
   step 1. Ask nothing about slots you just extracted or that are already filled.

Set "status" to REQUIREMENTS_COMPLETE only when questions is empty.
"rationale" is one sentence on what you extracted and what is still missing.
Do not answer the request. Do not produce a plan. Do not invent institutional data."""


# ---- Rule-based extraction, shared by the offline brain and used as a --------
# ---- safety net over the model's output (below). -----------------------------
def _extract_slots(text: str) -> Dict[str, Any]:
    # Collapse newlines/runs of spaces first, so a phrase split across two
    # lines ("placement\npercentage") still matches.
    text = re.sub(r"\s+", " ", text).strip()
    t = text.lower()
    out: Dict[str, Any] = {}

    # Word-boundary patterns, so "academic year 2025-26" does not silently
    # pull the Academic domain into scope.
    domain_patterns = {
        "Academic":  r"\bacademic(s)?\b(?!\s+year)|\bpass (percentage|rate)\b|\bco ?/? ?po\b",
        "Research":  r"\bresearch\b|\bpublication|\bcitation",
        "Placement": r"\bplacement|\binternship\b|\bavg package\b|\baverage package\b",
        "Finance":   r"\bfinance\b|\bfinancial\b|\bbudget\b|\brevenue\b",
    }
    domains = [d for d, pat in domain_patterns.items() if re.search(pat, t)]
    if re.search(r"\ball (four|4)\b|\ball domains\b|\bevery domain\b", t):
        domains = list(VALID_DOMAINS)
    if domains:
        out["domains"] = domains

    m = re.search(r"\b(20\d{2}\s*[-–/]\s*\d{2,4})\b", text) or re.search(
        r"\b(q[1-4]\s+20\d{2})\b", t)
    if m:
        out["period"] = ("academic year " + m.group(1).replace(" ", "")
                         if "20" in m.group(1) else m.group(1))

    if re.search(r"approved .*kpi|full kpi set|all (approved )?kpis|institution-wide kpi", t):
        out["kpis"] = ["approved institution-wide KPI set"]
    else:
        named = [n for n, pat in {
            "pass percentage": r"pass (percentage|%|rate)",
            "CO/PO attainment": r"co ?/? ?po",
            "publications": r"publication",
            "citations": r"citation",
            "grants": r"grant",
            "placement percentage": r"placement (percentage|%|rate)",
            "average package": r"average package|avg package",
            "internship completion": r"internship",
            "budget utilisation": r"budget utili",
            # Metrics leadership sometimes asks for that no linked source reports.
            # They are extracted faithfully here; the scope tool in Cell 5 is what
            # decides whether they can actually be sourced.
            "carbon footprint": r"carbon footprint",
            "alumni donations": r"alumni donation",
            "student satisfaction index": r"student satisfaction",
            "faculty retention": r"faculty retention",
        }.items() if re.search(pat, t)]
        if named:
            out["kpis"] = named

    targets = []
    if "nirf" in t:
        targets.append("NIRF")
    if "naac" in t:
        targets.append("NAAC")
    if re.search(r"strategic[- ]plan|strategic plan", t):
        targets.append("strategic plan")
    if targets:
        out["targets"] = targets

    if re.search(r"drill[- ]?down|department[- ]level", t):
        if re.search(r"only\b.*(miss|below|off|short of).{0,20}target", t):
            out["drilldown"] = "only for KPIs that miss their target"
        elif re.search(r"every kpi|all kpis|mandatory|each kpi", t):
            out["drilldown"] = "mandatory for every KPI"
        else:
            out["drilldown"] = "department-level drill-down required"

    if "dashboard" in t and ("brief" in t or "report" in t):
        out["output"] = "decision brief and executive dashboard"
    elif "dashboard" in t:
        out["output"] = "executive dashboard"
    elif re.search(r"decision brief|executive brief|leadership report|executive report", t):
        out["output"] = "decision brief with options and risks"
    return out


@llm.offline.register("clarify")
def _offline_clarify(payload: Dict[str, Any]) -> Dict[str, Any]:
    missing = payload["missing"]
    found = {k: v for k, v in _extract_slots(payload["request"]).items() if k in missing}
    still = [s for s in missing if s not in found]
    return {
        "status": "NEEDS_INFO" if still else "REQUIREMENTS_COMPLETE",
        "extracted": found,
        "questions": [SLOT_QUESTIONS[s] for s in still],
        "rationale": (f"Extracted {sorted(found) or 'nothing'} from the request; "
                      f"{len(still)} requirement(s) still unspecified."),
    }


def clarify(request: str, goal: "Goal") -> Dict[str, Any]:
    """Return {status, extracted, questions, rationale} for the empty slots only."""
    missing = goal.missing()
    if not missing:
        return {"status": "REQUIREMENTS_COMPLETE", "extracted": {}, "questions": [],
                "rationale": "All required slots were already filled."}

    filled = {s: getattr(goal, s) for s in REQUIRED_SLOTS if s not in missing}
    result = llm.json(
        task="clarify",
        system=CLARIFY_SYSTEM,
        prompt=CLARIFY_PROMPT.format(
            request=request.strip(),
            filled=json.dumps(filled, indent=2) if filled else "  (none yet)",
            missing="\n".join(f"  - {s}" for s in missing),
            question_bank="\n".join(f"  - {s}: {SLOT_QUESTIONS[s]}" for s in missing),
        ),
        schema=CLARIFY_SCHEMA,
        payload={"request": request, "missing": missing},
    )

    # ---- Guardrail: the model may only fill slots that were actually empty,
    # and may only propose domains from the approved roster. Anything else is
    # dropped rather than trusted.
    extracted = {k: v for k, v in (result.get("extracted") or {}).items()
                 if k in missing and v not in (None, "", [])}
    if "domains" in extracted:
        clean = [d for d in extracted["domains"]
                 if str(d).strip().title() in VALID_DOMAINS]
        if clean:
            extracted["domains"] = [str(d).strip().title() for d in clean]
        else:
            extracted.pop("domains")
    result["extracted"] = extracted

    # Re-derive the questions from state so they can never drift from reality.
    still = [s for s in missing if s not in extracted]
    if len(result.get("questions") or []) != len(still):
        result["questions"] = [SLOT_QUESTIONS[s] for s in still]
    result["status"] = "NEEDS_INFO" if still else "REQUIREMENTS_COMPLETE"
    result["asked_about"] = still
    return result


# ---- Quick check on the vague request the lab brief describes ---------------
_demo_goal = Goal()
_demo = clarify("Analyze our college performance.", _demo_goal)
print(f"status            : {_demo['status']}")
print(f"slots extracted   : {_demo['extracted'] or '{}  (nothing stated — nothing invented)'}")
print(f"questions asked   : {len(_demo['questions'])}  (acceptance criterion: >= 2)")
for i, q in enumerate(_demo["questions"], 1):
    print(f"   Q{i}. {q}")


status            : NEEDS_INFO
slots extracted   : {}  (nothing stated — nothing invented)
questions asked   : 6  (acceptance criterion: >= 2)
   Q1. Which performance domains must the brief cover — Academic, Research, Placement, Finance, or a subset?
   Q2. Which reporting period does this cover (e.g. academic year 2025-26, or a specific quarter)?
   Q3. Which KPIs should be reported, or should the approved institution-wide KPI set be used in full?
   Q4. Which target set should performance be compared against — NIRF, NAAC, the internal strategic plan, or all three?
   Q5. Is department-level drill-down required for every KPI, or only for KPIs that miss their target?
   Q6. What should leadership receive — a decision brief with options and risks, a dashboard, or both?


### Cell 5 · Act → Observe: a real tool call with a checked return

This is the line between a chatbot and an agent. A chatbot writes the plan from the prompt alone; an agent acts on the world and reads the result back before committing.

Chapter 15's execution methodology gates the run on Registrar-authorised read access, so before locking the plan the coordinator reads the **data-scope register** and plans around what it actually finds there — including figures that are provisional rather than final, and requested KPIs that no linked system reports.

The tool returns `{"register": ...}` **or** `{"error": ...}`, and the caller checks. In Lab 5 this same read moves behind the MCP-style connector; the call signature will not change.

In [5]:
# =============================================================================
# Cell 5  ·  The Act -> Observe step: a real tool call, with a checked return
# =============================================================================
# Lab 1 is "Agent vs Chatbot". A chatbot writes a plan from the prompt alone.
# An agent *acts on the world and reads the result back* before committing.
#
# Gate 1 of the Chapter 15 hybrid execution methodology says the run may only
# start "after each domain grants institution-level read access authorized by
# the Registrar". So before the plan is locked, the coordinator calls the data
# scope register and plans around what it actually observes there.
#
# In Lab 5 this same read moves behind the MCP-style connector; the call
# signature stays the same, which is exactly why it is written as a tool now.

DATA_SCOPE_REGISTER = {
    "Academic": {
        "source_system": "Academic ERP — examination & OBE records",
        "read_access": "granted by Registrar on 2026-06-01",
        "periods": ["2023-24", "2024-25", "2025-26"],
        "figure_status": "final",
        "metrics": ["pass percentage", "CO/PO attainment", "curriculum compliance"],
    },
    "Research": {
        "source_system": "Research & Publications repository (Scopus-synced)",
        "read_access": "granted by Registrar on 2026-06-01",
        "periods": ["2023-24", "2024-25", "2025-26"],
        "figure_status": "final",
        "metrics": ["publications", "citations", "grants sanctioned"],
    },
    "Placement": {
        "source_system": "Placement Cell tracker",
        "read_access": "granted by Registrar on 2026-06-04",
        "periods": ["2023-24", "2024-25", "2025-26"],
        "figure_status": "final",
        "metrics": ["placement percentage", "average package", "internship completion"],
        "caveat": "3 provisional offers are excluded until the Placement Cell verifies them.",
    },
    "Finance": {
        "source_system": "Finance & Accounts ledger",
        "read_access": "granted by Registrar on 2026-06-07",
        "periods": ["2023-24", "2024-25"],          # 2025-26 not closed yet
        "figure_status": "provisional",
        "metrics": ["budget utilisation", "revenue", "grant funds"],
        "caveat": "Q3-Q4 of 2025-26 are unreconciled; the Finance Office closes books in August.",
    },
}


def read_data_scope_register(domains: List[str], period: str) -> Dict[str, Any]:
    """TOOL. Returns the access + coverage record for the requested domains.

    Contract: returns either {"register": ...} or {"error": ...}. The caller
    must check — a tool result is never assumed to be good.
    """
    unlinked = [d for d in domains if d not in DATA_SCOPE_REGISTER]
    if unlinked:
        return {"error": f"no source system linked for domain(s): {unlinked}"}
    return {
        "register": {d: DATA_SCOPE_REGISTER[d] for d in domains},
        "period_requested": period,
        "checked_at": "run-time",
    }


def detect_conflicts(goal: "Goal", register: Dict[str, Any], period: str) -> List[Dict[str, Any]]:
    """Compare what leadership ASKED FOR against what the register actually offers.

    Silently planning over provisional data is the failure mode the Chapter 15
    Finance-Agent guardrail exists to prevent ("never project a forecast as an
    actual"), so every mismatch is raised explicitly instead of absorbed.
    """
    conflicts: List[Dict[str, Any]] = []
    period_key = re.sub(r"^academic year\s*", "", period.strip(), flags=re.I)

    for domain, rec in register.items():
        # (a) requested period is outside what this source can serve
        if period_key not in rec["periods"]:
            conflicts.append({
                "type": "PERIOD_NOT_COVERED",
                "domain": domain,
                "detail": (f"{domain} source covers {rec['periods']}, "
                           f"but the request asks for {period_key}."),
                "required_action": (f"Either exclude {domain} from this run, or report it "
                                    f"against the last closed period with an explicit label."),
            })
        # (b) figures exist but are not final
        if rec["figure_status"] != "final":
            conflicts.append({
                "type": "FIGURES_NOT_FINAL",
                "domain": domain,
                "detail": f"{domain} figures are '{rec['figure_status']}'. {rec.get('caveat','')}".strip(),
                "required_action": ("Label every figure from this domain as provisional in the "
                                    "brief, or drop the domain until reconciliation completes."),
            })
        elif rec.get("caveat"):
            conflicts.append({
                "type": "SCOPE_CAVEAT",
                "domain": domain,
                "detail": rec["caveat"],
                "required_action": "Record the caveat in the brief's assumptions section.",
            })

    # (c) a requested KPI that no linked source can supply
    if goal.kpis and goal.kpis != ["approved institution-wide KPI set"]:
        available = {m.lower() for rec in register.values() for m in rec["metrics"]}
        for kpi in goal.kpis:
            if not any(kpi.lower() in m or m in kpi.lower() for m in available):
                conflicts.append({
                    "type": "KPI_NOT_SOURCED",
                    "domain": "-",
                    "detail": f"No linked source system reports '{kpi}'.",
                    "required_action": f"Drop '{kpi}' from scope or authorise a new source system.",
                })
    return conflicts


# ---- Smoke test: ask for all four domains in 2025-26 and see what comes back -
_probe = read_data_scope_register(VALID_DOMAINS, "academic year 2025-26")
assert "error" not in _probe, _probe
_g = Goal(kpis=["placement percentage", "carbon footprint"])
for c in detect_conflicts(_g, _probe["register"], "academic year 2025-26"):
    print(f"[{c['type']:<19}] {c['domain']:<10} {c['detail']}")


[SCOPE_CAVEAT       ] Placement  3 provisional offers are excluded until the Placement Cell verifies them.
[PERIOD_NOT_COVERED ] Finance    Finance source covers ['2023-24', '2024-25'], but the request asks for 2025-26.
[FIGURES_NOT_FINAL  ] Finance    Finance figures are 'provisional'. Q3-Q4 of 2025-26 are unreconciled; the Finance Office closes books in August.
[KPI_NOT_SOURCED    ] -          No linked source system reports 'carbon footprint'.


### Cell 6 · `propose_plan()` — an editable plan object

The acceptance line asks for a *coherent, editable* plan. Prose is not editable by a program — you cannot remove a domain from a paragraph without rewriting the paragraph. So the plan is a typed object whose steps carry stable ids (`S1`, `S2`, …). That single decision is what makes the mutating `revise()` in the next cell possible at all.

After the model drafts, Python enforces the invariants it is not allowed to lose: every observed conflict must appear in the assumptions, and the four Chapter 15 validation checks must be present.

In [6]:
# =============================================================================
# Cell 6  ·  propose_plan()  —  an editable PLAN OBJECT, not a wall of prose
# =============================================================================
# The Lab 1 acceptance line for Worklet 15 asks for "a coherent, editable plan".
# Prose is not editable by a program: you cannot remove a domain from a
# paragraph without regenerating the paragraph. So the plan is a typed object
# with addressable parts (steps carry IDs), which is what makes the mutating
# revise() in the next cell possible at all.

PLAN_SCHEMA = {
    "type": "object",
    "properties": {
        "objective": {"type": "string"},
        "scope": {
            "type": "object",
            "properties": {
                "domains": {"type": "array", "items": {"type": "string"}},
                "period": {"type": "string"},
                "unit_of_analysis": {"type": "string"},
            },
        },
        "data_required": {
            "type": "array",
            "items": {"type": "object", "properties": {
                "domain": {"type": "string"}, "source_system": {"type": "string"},
                "fields": {"type": "array", "items": {"type": "string"}},
                "access": {"type": "string"}}},
        },
        "analysis_steps": {
            "type": "array",
            "items": {"type": "object", "properties": {
                "id": {"type": "string"}, "title": {"type": "string"},
                "detail": {"type": "string"}, "owner_agent": {"type": "string"}}},
        },
        "kpi_calculations": {
            "type": "array",
            "items": {"type": "object", "properties": {
                "kpi": {"type": "string"}, "formula": {"type": "string"},
                "unit": {"type": "string"}, "source": {"type": "string"}}},
        },
        "target_comparison": {"type": "object", "properties": {
            "target_sets": {"type": "array", "items": {"type": "string"}},
            "method": {"type": "string"}}},
        "drilldown_policy": {"type": "string"},
        "evidence_requirements": {"type": "array", "items": {"type": "string"}},
        "brief_structure": {"type": "array", "items": {"type": "string"}},
        "validation_checks": {"type": "array", "items": {"type": "string"}},
        "assumptions": {"type": "array", "items": {"type": "string"}},
    },
    "required": ["objective", "scope", "data_required", "analysis_steps",
                 "kpi_calculations", "target_comparison", "drilldown_policy",
                 "evidence_requirements", "brief_structure", "validation_checks",
                 "assumptions"],
}

PLAN_SYSTEM = """You are the planning component of Worklet 15, an Executive Decision
Support System. You produce execution PLANS only. You never compute a KPI, never
state an institutional figure, and never write the decision brief itself."""

PLAN_PROMPT = """Produce an execution plan from the locked requirements below, and from
nothing else.

REQUIREMENTS (all six slots confirmed with the user):
{requirements}

OBSERVED DATA SCOPE (returned by the data-scope register tool — this is real,
use it):
{scope}

DATA CONFLICTS the tool call surfaced (each one MUST be reflected in the plan,
either as an assumption or as an explicit scope exclusion):
{conflicts}

Rules:
- Do not invent institutional data. Do not compute results. Do not write the brief.
- Write NO numeric figure anywhere: no percentages, no counts, no amounts, no target
  values. Lab 1 produces a plan, not results. Naming a metric is fine ("pass
  percentage"); stating a value for it ("92%") is not.
- Every analysis step gets a short stable id: S1, S2, S3, ...
- Every KPI calculation names its formula and its source system.
- validation_checks must include the four Chapter 15 acceptance checks:
  every figure cites its source; assumptions are stated explicitly; each option
  lists its risks and trade-offs; no action is taken without leadership approval.
- brief_structure is the section list of the final decision brief.
The plan must be precise enough for another agent to execute without asking anything."""


@llm.offline.register("propose_plan")
def _offline_plan(payload: Dict[str, Any]) -> Dict[str, Any]:
    g, conflicts, register = payload["goal"], payload["conflicts"], payload["register"]
    domains, period = g["domains"], g["period"]

    kpi_catalog = {
        "Academic":  [("Pass percentage", "passed_students / appeared_students x 100", "%"),
                      ("CO/PO attainment", "mean(attainment_level) over mapped COs", "index 0-3")],
        "Research":  [("Publications per faculty", "publications / sanctioned_faculty", "count"),
                      ("Citations per faculty", "citations / sanctioned_faculty", "count"),
                      ("Grants sanctioned", "sum(grant_amount)", "INR lakh")],
        "Placement": [("Placement percentage", "placed_students / eligible_students x 100", "%"),
                      ("Average package", "sum(package) / placed_students", "LPA"),
                      ("Internship completion", "interns_completed / eligible_students x 100", "%")],
        "Finance":   [("Budget utilisation", "spent / allocated x 100", "%"),
                      ("Grant funds received", "sum(grant_credits)", "INR lakh")],
    }
    kpis = [{"kpi": k, "formula": f, "unit": u,
             "source": register.get(d, {}).get("source_system", "-")}
            for d in domains for (k, f, u) in kpi_catalog.get(d, [])]

    steps = [{"id": "S1", "title": "Confirm read access and freeze the data cut",
              "detail": ("Re-read the data-scope register for "
                         f"{', '.join(domains)} and record the cut timestamp so every figure "
                         "in the brief is reproducible."),
              "owner_agent": "Coordinator"}]
    for i, d in enumerate(domains, start=2):
        steps.append({
            "id": f"S{i}",
            "title": f"Aggregate {d.lower()} metrics across departments",
            "detail": (f"{d} Agent pulls {', '.join(register.get(d, {}).get('metrics', []))} "
                       f"per department for {period}, keeps the per-department spread, and "
                       "flags any department with stale or missing data."),
            "owner_agent": f"{d} Agent"})
    n = len(steps)
    steps += [
        {"id": f"S{n+1}", "title": "Synthesise institution-wide KPIs",
         "detail": ("KPI Agent applies only the approved KPI definitions to the domain "
                    "aggregates, showing formula and source for each; enrolment-weighted, "
                    "never a plain mean of department percentages."),
         "owner_agent": "KPI Agent"},
        {"id": f"S{n+2}", "title": "Compare each KPI to its approved target",
         "detail": (f"Compute variance against {', '.join(g['targets'])}; classify each KPI as "
                    "on-target / at-risk / below-target using the approved thresholds."),
         "owner_agent": "KPI Agent"},
        {"id": f"S{n+3}", "title": "Attach department-level drill-down evidence",
         "detail": ("For each KPI, record the traceable path KPI -> contributing department -> "
                    "source metric -> source system, so leadership can trace any number back."),
         "owner_agent": "Report Generation Agent"},
        {"id": f"S{n+4}", "title": "Draft options and risks for every below-target KPI",
         "detail": ("For each KPI missing target, draft 2-3 options with cost, owner, timeline "
                    "and the risks/trade-offs of each. Options only — no action is taken."),
         "owner_agent": "Report Generation Agent"},
        {"id": f"S{n+5}", "title": "Run validation, then route to the human gate",
         "detail": ("Validation is a hard gate; on failure only the failing KPI is regenerated. "
                    "The Principal/Registrar then approves, edits or overrides before circulation."),
         "owner_agent": "Coordinator + Executive gate"},
    ]

    assumptions = [
        f"Analysis is limited to {period}; prior years appear only as trend context.",
        "Only Registrar-authorised, institution-level read access is used; no department is queried directly.",
        "KPI definitions and target values are the approved set — this plan does not redefine them.",
    ] + [f"{c['type']}: {c['detail']} -> {c['required_action']}" for c in conflicts]

    return {
        "objective": (f"Produce a leadership decision brief comparing {', '.join(domains)} "
                      f"performance for {period} against {', '.join(g['targets'])}, "
                      "identifying below-target areas with options and risks."),
        "scope": {"domains": domains, "period": period,
                  "unit_of_analysis": "institution level, with department-level drill-down"},
        "data_required": [{"domain": d,
                           "source_system": register.get(d, {}).get("source_system", "-"),
                           "fields": register.get(d, {}).get("metrics", []),
                           "access": register.get(d, {}).get("read_access", "-")} for d in domains],
        "analysis_steps": steps,
        "kpi_calculations": kpis,
        "target_comparison": {"target_sets": g["targets"],
                              "method": ("variance = actual - target, reported in the KPI's own unit "
                                         "alongside % of target achieved; no rounding may hide a miss")},
        "drilldown_policy": g["drilldown"],
        "evidence_requirements": [
            "Every figure carries source system, extraction timestamp and department scope.",
            "Every KPI carries its formula and the approved target it is measured against.",
            "Provisional or unreconciled figures are labelled inline, never presented as final.",
            "Every option carries its risks, trade-offs and named owner.",
        ],
        "brief_structure": [
            "1. Executive summary (3 bullets, decision-ready)",
            "2. KPI scorecard vs targets, with variance",
            "3. Below-target KPIs with department-level drill-down",
            "4. Options with cost, owner, timeline",
            "5. Risks and trade-offs per option",
            "6. Assumptions and data caveats",
            "7. Evidence appendix (source trace per figure)",
        ],
        "validation_checks": [
            "every figure cites its source",
            "assumptions are stated explicitly",
            "each option lists its risks and trade-offs",
            "no action is taken without leadership approval",
        ],
        "assumptions": assumptions,
    }


@dataclass
class Plan:
    """A mutable, addressable plan. revise() edits THIS object in place."""
    body: Dict[str, Any]
    revision: int = 0
    patch_history: List[Dict[str, Any]] = field(default_factory=list)

    def to_dict(self) -> Dict[str, Any]:
        return {"revision": self.revision, **self.body,
                "patch_history": self.patch_history}

    def render(self) -> str:
        b, out = self.body, [f"EXECUTION PLAN  (revision {self.revision})", "=" * 72]
        out += [f"OBJECTIVE\n  {textwrap.fill(b.get('objective', '-'), 88, subsequent_indent='  ')}"]
        s = b.get("scope", {})
        out += [f"\nSCOPE\n  domains : {', '.join(s.get('domains', []))}"
                f"\n  period  : {s.get('period', '-')}"
                f"\n  unit    : {s.get('unit_of_analysis', '-')}"]
        out += ["\nDATA REQUIRED"] + [
            f"  - {d.get('domain', '-'):<10} {d.get('source_system', '-')}\n"
            f"    fields: {', '.join(d.get('fields', []))}"
            for d in b.get("data_required", [])]
        out += ["\nANALYSIS STEPS"] + [
            f"  [{st.get('id', '?')}] {st.get('title', '-')}  ({st.get('owner_agent', '-')})\n"
            + textwrap.fill(st.get("detail", ""), 88,
                            initial_indent="        ", subsequent_indent="        ")
            for st in b.get("analysis_steps", [])]
        out += ["\nKPI CALCULATIONS"] + [
            f"  - {k.get('kpi', '-'):<26} = {k.get('formula', '-')}  [{k.get('unit', '-')}]"
            for k in b.get("kpi_calculations", [])]
        t = b.get("target_comparison", {})
        out += [f"\nTARGET COMPARISON\n  against : {', '.join(t.get('target_sets', []))}"
                f"\n  method  : {t.get('method', '-')}"]
        out += [f"\nDRILL-DOWN POLICY\n  {b.get('drilldown_policy', '-')}"]
        if b.get("brief_length_limit"):
            out += [f"\nLENGTH LIMIT\n  {b['brief_length_limit']}"]
        for label, key in [("EVIDENCE REQUIREMENTS", "evidence_requirements"),
                           ("DECISION BRIEF STRUCTURE", "brief_structure"),
                           ("VALIDATION CHECKS (hard gate)", "validation_checks"),
                           ("ASSUMPTIONS & DATA CAVEATS", "assumptions")]:
            out += [f"\n{label}"] + [
                textwrap.fill(f"  - {x}", 90, subsequent_indent="    ")
                for x in b.get(key, [])]
        if self.patch_history:
            out += ["\nREVISION HISTORY"] + [
                textwrap.fill(f"  r{p['revision']}: {p['op']} — {p['effect']}", 90,
                              subsequent_indent="        ") for p in self.patch_history]
        if b.get("_unchanged_note"):
            out += [f"\nLAST REVISION LEFT UNTOUCHED\n  "
                    + textwrap.fill(b['_unchanged_note'], 88, subsequent_indent='  ')]
        return "\n".join(out)


def propose_plan(goal: "Goal", register: Dict[str, Any],
                 conflicts: List[Dict[str, Any]]) -> Plan:
    body = llm.json(
        task="propose_plan",
        system=PLAN_SYSTEM,
        prompt=PLAN_PROMPT.format(
            requirements=json.dumps(
                {s: getattr(goal, s) for s in REQUIRED_SLOTS}, indent=2),
            scope=json.dumps(register, indent=2),
            conflicts=json.dumps(conflicts, indent=2) if conflicts else "  (none)"),
        schema=PLAN_SCHEMA,
        payload={"goal": {s: getattr(goal, s) for s in REQUIRED_SLOTS},
                 "conflicts": conflicts, "register": register},
    )

    # ---- Normalisation: a model may omit an optional nested field. Fill the
    # ---- shape in from the confirmed requirements before anything reads it.
    for key, default in [("objective", ""), ("scope", {}), ("data_required", []),
                         ("analysis_steps", []), ("kpi_calculations", []),
                         ("target_comparison", {}), ("drilldown_policy", ""),
                         ("evidence_requirements", []), ("brief_structure", []),
                         ("validation_checks", []), ("assumptions", [])]:
        if not isinstance(body.get(key), type(default)):
            body[key] = default if not isinstance(default, (dict, list)) else type(default)()
    body["scope"].setdefault("domains", list(goal.domains or []))
    body["scope"].setdefault("period", goal.period or "")
    body["scope"].setdefault("unit_of_analysis",
                             "institution level, with department-level drill-down")
    body["target_comparison"].setdefault("target_sets", list(goal.targets or []))
    body["target_comparison"].setdefault("method", "variance against the approved target")
    body["drilldown_policy"] = body.get("drilldown_policy") or (goal.drilldown or "")
    for d in body["data_required"]:
        d.setdefault("domain", "-")
        d.setdefault("source_system", register.get(d.get("domain", ""), {})
                     .get("source_system", "-"))
        d.setdefault("fields", register.get(d.get("domain", ""), {}).get("metrics", []))
        d.setdefault("access", register.get(d.get("domain", ""), {}).get("read_access", "-"))
    for st in body["analysis_steps"]:
        st.setdefault("title", "step")
        st.setdefault("detail", "")
        st.setdefault("owner_agent", "Coordinator")

    # ---- Invariants Python enforces, so a model can never quietly drop them --
    for c in conflicts:                       # every conflict must survive into the plan
        line = f"{c['type']}: {c['detail']} -> {c['required_action']}"
        if not any(c["type"] in a for a in body["assumptions"]):
            body["assumptions"].append(line)
    required_checks = ["every figure cites its source", "assumptions are stated explicitly",
                       "each option lists its risks and trade-offs",
                       "no action is taken without leadership approval"]
    have = " ".join(body.get("validation_checks", [])).lower()
    body["validation_checks"] = list(body.get("validation_checks", [])) + [
        c for c in required_checks if c.split()[1] not in have]
    for i, st in enumerate(body.get("analysis_steps", []), 1):   # stable ids for revise()
        st.setdefault("id", f"S{i}")
    return Plan(body=body)

print("propose_plan() ready — returns a Plan object with addressable steps (S1, S2, ...).")


propose_plan() ready — returns a Plan object with addressable steps (S1, S2, ...).


### Cell 7 · `revise()` — mutate, never regenerate

> *"`revise()` mutates the existing plan object on feedback rather than calling `run()` again from scratch — this is the single detail instructors should check first, since regenerating from zero is the most common way students accidentally fail this lab's rubric."* — Lab Guide, Lab 1

So the model is **not permitted to rewrite the plan**. It may only emit a typed **patch**: a list of operations, each with a `why`. Python applies them to the live object.

Three consequences, all verified in Cell 13:

* everything the user did not comment on is *provably* untouched (field-level diff),
* every edit is auditable (`patch_history` records op → why → effect), and
* `id(plan)` is stable across every revision.

In [7]:
# =============================================================================
# Cell 7  ·  revise()  —  MUTATE the existing plan; never regenerate it
# =============================================================================
# This is the single detail the lab guide tells instructors to check first:
# "revise() mutates the existing plan object on feedback rather than calling
#  run() again from scratch ... regenerating from zero is the most common way
#  students accidentally fail this lab's rubric."
#
# So the model is NOT allowed to rewrite the plan. It may only emit a typed
# PATCH — a list of operations — which Python applies to the live object.
# Consequences:
#   * everything the user did not comment on is provably untouched,
#   * every edit is auditable (patch_history), and
#   * id(plan) is stable across revisions, which the checks cell asserts.

PATCH_SCHEMA = {
    "type": "object",
    "properties": {
        "ops": {"type": "array", "items": {"type": "object", "properties": {
            "op": {"type": "string", "enum": ["remove_domain", "add_domain", "set_field",
                                              "add_item", "remove_item", "modify_step",
                                              "add_step", "remove_step"]},
            "path": {"type": "string"},
            "value": {"type": "string"},
            "step_id": {"type": "string"},
            "after": {"type": "string"},
            "title": {"type": "string"},
            "owner_agent": {"type": "string"},
            "why": {"type": "string"},
        }, "required": ["op", "why"]}},
        "unchanged_note": {"type": "string"},
    },
    "required": ["ops", "unchanged_note"],
}

REVISE_SYSTEM = """You revise execution plans for Worklet 15 by emitting PATCH OPERATIONS
only. You never restate or rewrite the plan. Anything the feedback does not mention must
be left completely untouched."""

REVISE_PROMPT = """CURRENT PLAN (revision {revision}):
{plan}

USER FEEDBACK:
{feedback}

Emit the minimal list of operations that applies this feedback, and nothing more.

Available operations:
  remove_domain  value=<Academic|Research|Placement|Finance>
  add_domain     value=<domain>
  set_field      path=<dotted path, e.g. drilldown_policy or scope.period>  value=<new value>
  add_item       path=<list field, e.g. evidence_requirements>              value=<new entry>
  remove_item    path=<list field>  value=<substring identifying the entry>
  modify_step    step_id=<S3>  value=<replacement detail>
  add_step       after=<S3>  title=<...>  value=<detail>  owner_agent=<...>
  remove_step    step_id=<S5>

Every operation carries "why" — one short sentence quoting the part of the feedback it
serves. Do not add operations the user did not ask for. Do not reword the objective
unless the feedback asks you to. "unchanged_note" states what you deliberately left alone."""


@llm.offline.register("revise")
def _offline_revise(payload: Dict[str, Any]) -> Dict[str, Any]:
    fb, ops = payload["feedback"].lower(), []

    for d in VALID_DOMAINS:
        if re.search(rf"(remove|drop|exclude|without)\b[^.]*\b{d.lower()}", fb):
            ops.append({"op": "remove_domain", "value": d,
                        "why": f"feedback asks to remove the {d} domain"})
        elif re.search(rf"(add|include|also cover)\b[^.]*\b{d.lower()}", fb):
            ops.append({"op": "add_domain", "value": d,
                        "why": f"feedback asks to add the {d} domain"})

    if re.search(r"drill[- ]?down", fb) and re.search(r"every|all kpi|mandatory|each kpi", fb):
        ops.append({"op": "set_field", "path": "drilldown_policy",
                    "value": ("mandatory for every KPI — each KPI must carry its full "
                              "department-level trace, not only the KPIs that miss target"),
                    "why": "feedback makes department-level drill-down mandatory for every KPI"})

    m = re.search(r"\b(one|two|three|1|2|3)[- ]page", fb)
    if m:
        word = {"1": "one", "2": "two", "3": "three"}.get(m.group(1), m.group(1))
        ops.append({"op": "set_field", "path": "brief_length_limit",
                    "value": (f"maximum {word} pages for the main brief; the evidence "
                              "appendix ships as a separate annexure"),
                    "why": "feedback caps the length of the main brief"})

    if re.search(r"risk owner|named owner|owner for (each|every)|accountab", fb):
        ops.append({"op": "add_item", "path": "evidence_requirements",
                    "value": ("Every option names an accountable owner and the risk that "
                              "owner is accepting."),
                    "why": "feedback requires an accountable owner against every option"})

    if re.search(r"trend|year[- ]on[- ]year|yoy|previous year|last year|3[- ]year", fb):
        ops.append({"op": "add_step", "after": "S1",
                    "title": "Pull the prior-year baseline for trend context",
                    "value": ("Retrieve the same KPI set for the preceding closed year so every "
                              "KPI is reported with its year-on-year movement, clearly labelled "
                              "as trend context rather than as the reporting-period figure."),
                    "owner_agent": "Coordinator",
                    "why": "feedback asks for year-on-year trend context"})

    if re.search(r"executive summary|shorter summary|top line", fb) and "remove" in fb:
        ops.append({"op": "remove_item", "path": "brief_structure", "value": "Executive summary",
                    "why": "feedback removes the executive summary section"})

    return {"ops": ops,
            "unchanged_note": ("Objective, data sources, KPI formulas, target-comparison method "
                               "and validation checks were left untouched.")}


# ---- The patch applier: the only code allowed to change a Plan --------------
def _get_parent(body: Dict[str, Any], path: str):
    parts, cur = path.split("."), body
    for p in parts[:-1]:
        cur = cur.setdefault(p, {})
    return cur, parts[-1]


def apply_patch(plan: Plan, ops: List[Dict[str, Any]]) -> List[str]:
    """Apply operations IN PLACE to `plan`. Returns human-readable effects."""
    b, effects = plan.body, []
    b.setdefault("scope", {}).setdefault("domains", [])
    for key in ("data_required", "kpi_calculations", "analysis_steps", "assumptions"):
        b.setdefault(key, [])

    for op in ops:
        kind = op.get("op")

        if kind in ("remove_domain", "add_domain"):
            dom = str(op.get("value", "")).strip().title()
            if dom not in VALID_DOMAINS:
                effects.append(f"IGNORED {kind}: '{op.get('value')}' is not an approved domain")
                continue
            if kind == "remove_domain":
                if dom not in b["scope"]["domains"]:
                    effects.append(f"IGNORED remove_domain: {dom} is not in scope")
                    continue
                b["scope"]["domains"].remove(dom)
                b["data_required"] = [d for d in b["data_required"] if d["domain"] != dom]
                src = DATA_SCOPE_REGISTER.get(dom, {}).get("source_system", "~none~")
                b["kpi_calculations"] = [k for k in b["kpi_calculations"] if k.get("source") != src]
                dropped = [s["id"] for s in b["analysis_steps"]
                           if s.get("owner_agent", "").startswith(dom)]
                b["analysis_steps"] = [s for s in b["analysis_steps"] if s["id"] not in dropped]
                # Conflicts about a removed domain are retired, not deleted — audit trail.
                b["assumptions"] = [
                    (f"[retired at r{plan.revision + 1}: {dom} removed from scope] {a}"
                     if (dom in a and a.split(":")[0].isupper()) else a)
                    for a in b["assumptions"]]
                effects.append(f"removed domain {dom}: dropped steps {dropped or '[]'}, "
                               f"its data source and its KPI rows")
            else:
                if dom in b["scope"]["domains"]:
                    effects.append(f"IGNORED add_domain: {dom} already in scope")
                    continue
                b["scope"]["domains"].append(dom)
                rec = DATA_SCOPE_REGISTER.get(dom, {})
                b["data_required"].append({"domain": dom,
                                           "source_system": rec.get("source_system", "-"),
                                           "fields": rec.get("metrics", []),
                                           "access": rec.get("read_access", "-")})
                effects.append(f"added domain {dom} with its source system")

        elif kind == "set_field":
            parent, leaf = _get_parent(b, op["path"])
            old = parent.get(leaf)
            parent[leaf] = op["value"]
            effects.append(f"set {op['path']} := '{op['value']}'"
                           + (f" (was '{old}')" if old is not None else " (new field)"))

        elif kind == "add_item":
            parent, leaf = _get_parent(b, op["path"])
            parent.setdefault(leaf, []).append(op["value"])
            effects.append(f"appended to {op['path']}: '{op['value'][:60]}...'")

        elif kind == "remove_item":
            parent, leaf = _get_parent(b, op["path"])
            before = list(parent.get(leaf, []))
            parent[leaf] = [x for x in before if op["value"].lower() not in str(x).lower()]
            effects.append(f"removed {len(before) - len(parent[leaf])} entry(ies) from {op['path']}")

        elif kind == "modify_step":
            for st in b["analysis_steps"]:
                if st["id"] == op.get("step_id"):
                    st["detail"] = op["value"]
                    effects.append(f"rewrote detail of step {st['id']}")
                    break
            else:
                effects.append(f"IGNORED modify_step: no step {op.get('step_id')}")

        elif kind == "remove_step":
            n = len(b["analysis_steps"])
            b["analysis_steps"] = [s for s in b["analysis_steps"] if s["id"] != op.get("step_id")]
            effects.append(f"removed step {op.get('step_id')}"
                           if len(b["analysis_steps"]) < n else
                           f"IGNORED remove_step: no step {op.get('step_id')}")

        elif kind == "add_step":
            ids = [s["id"] for s in b["analysis_steps"]]
            new_id = f"S{max([int(i[1:]) for i in ids if i[1:].isdigit()] + [0]) + 1}"
            new = {"id": new_id, "title": op.get("title", "additional step"),
                   "detail": op.get("value", ""), "owner_agent": op.get("owner_agent", "Coordinator")}
            pos = ids.index(op["after"]) + 1 if op.get("after") in ids else len(ids)
            b["analysis_steps"].insert(pos, new)
            effects.append(f"inserted step {new_id} after {op.get('after', 'end')}: {new['title']}")

        else:
            effects.append(f"IGNORED unknown op: {kind}")

    return effects


def revise(plan: Plan, feedback: str) -> Plan:
    """Apply feedback to the SAME Plan object. Returns the same instance."""
    patch = llm.json(task="revise", system=REVISE_SYSTEM,
                     prompt=REVISE_PROMPT.format(revision=plan.revision,
                                                 plan=json.dumps(plan.body, indent=2),
                                                 feedback=feedback.strip()),
                     schema=PATCH_SCHEMA, payload={"feedback": feedback, "plan": plan.body})
    ops = patch.get("ops", [])
    effects = apply_patch(plan, ops)                   # <-- in-place mutation
    plan.revision += 1
    for op, eff in zip(ops, effects):
        plan.patch_history.append({"revision": plan.revision, "op": op.get("op"),
                                   "why": op.get("why", ""), "effect": eff})
    if not ops:
        plan.patch_history.append({"revision": plan.revision, "op": "none",
                                   "why": "no actionable change found in the feedback",
                                   "effect": "plan left unchanged"})
    plan.body["_unchanged_note"] = patch.get("unchanged_note", "")
    return plan                                        # same object, higher revision

print("revise() ready — emits typed patch ops and applies them in place.")
print("Supported ops:", ", ".join(PATCH_SCHEMA["properties"]["ops"]["items"]["properties"]["op"]["enum"]))


revise() ready — emits typed patch ops and applies them in place.
Supported ops: remove_domain, add_domain, set_field, add_item, remove_item, modify_step, add_step, remove_step


### Cell 8 · The Coordinator

One loop, driven entirely by goal state. `next_step()` inspects what is missing and picks the next action — it is not a fixed questionnaire.

Every iteration appends `{planned, action, observed, stopped}` (plus a `goal_state` snapshot and, at a stop, a `stop_reason`) to `self.log`.

The clarification budget matters: after `MAX_CLARIFY_ROUNDS` the agent **stops and says the requirements are incomplete** rather than inventing them. Check A16 exercises that path.

In [8]:
# =============================================================================
# Cell 8  ·  The Coordinator — Goal -> Plan -> Act -> Observe -> Revise
# =============================================================================
# One loop, driven entirely by goal state. `next_step()` looks at what is still
# missing and picks the next action; it is not a fixed questionnaire, which the
# second demo run below proves by skipping straight past questions the request
# already answered.
#
# Every iteration appends {planned, action, observed, stopped} to the log —
# the "prompt log" the Lab 1 deliverable asks for.

MAX_CLARIFY_ROUNDS = 4


class CoordinatorAgent:
    """apniLeap AURA Coordinator, Lab 1 scope: loop only, no specialist agents."""

    def __init__(self, answer_provider):
        self.goal = Goal()
        self.plan: Optional[Plan] = None
        self.register: Dict[str, Any] = {}
        self.conflicts: List[Dict[str, Any]] = []
        self.log: List[Dict[str, Any]] = []
        self.context = ""                    # request + every user reply so far
        self.answer_provider = answer_provider
        self.rounds = 0

    # ---- logging -------------------------------------------------------
    def _log(self, planned, action, observed, stopped=False, stop_reason=None):
        entry = {"iteration": len(self.log) + 1, "planned": planned, "action": action,
                 "observed": observed, "stopped": stopped,
                 "goal_state": {"missing": self.goal.missing(),
                                "scope_checked": self.goal.scope_checked,
                                "plan_locked": self.goal.plan_locked}}
        if stop_reason:
            entry["stop_reason"] = stop_reason
        self.log.append(entry)
        return entry

    # ---- planner: what do we do next? ----------------------------------
    def next_step(self):
        if self.goal.missing():
            return ("clarify", self.goal.missing())
        if not self.goal.scope_checked:
            return ("check_scope", None)
        return ("generate_plan", None)

    # ---- the loop -------------------------------------------------------
    def run(self, request: str) -> Optional[Plan]:
        self.context = request.strip()
        self._log(planned="ingest the leadership request",
                  action="received free-text request",
                  observed=self.context)

        while not self.goal.satisfied():                       # <-- stopping condition
            step, arg = self.next_step()

            # ---------- CLARIFY (ask only what is missing) ----------------
            if step == "clarify":
                self.rounds += 1
                before = set(REQUIRED_SLOTS) - set(self.goal.missing())
                result = clarify(self.context, self.goal)
                for slot, value in result["extracted"].items():
                    setattr(self.goal, slot, value)
                gained = sorted((set(REQUIRED_SLOTS) - set(self.goal.missing())) - before)

                self._log(planned=f"clarify {len(arg)} unspecified requirement(s): {arg}",
                          action=(f"round {self.rounds}: extracted {gained or 'nothing'} from the "
                                  f"request text; asked {len(result['questions'])} question(s)"),
                          observed={"questions": result["questions"],
                                    "extracted": result["extracted"],
                                    "status": result["status"]})

                if result["questions"]:
                    if self.rounds > MAX_CLARIFY_ROUNDS:
                        self._log(planned="stop rather than assume",
                                  action="clarification budget exhausted",
                                  observed={"still_missing": self.goal.missing()},
                                  stopped=True,
                                  stop_reason=("requirements still incomplete after "
                                               f"{MAX_CLARIFY_ROUNDS} rounds; the agent refuses "
                                               "to invent the missing slots"))
                        return None
                    reply = self.answer_provider(result["questions"], self.rounds)
                    if reply is None:
                        self._log(planned="stop rather than assume", action="no reply available",
                                  observed={"still_missing": self.goal.missing()}, stopped=True,
                                  stop_reason="user did not supply the missing requirements")
                        return None
                    self.context += "\n\nUSER REPLY:\n" + reply.strip()
                    self._log(planned="fold the reply into the run context and re-check",
                              action=f"received reply to round {self.rounds}",
                              observed=reply.strip())

            # ---------- ACT + OBSERVE (the real tool call) ----------------
            elif step == "check_scope":
                tool_out = read_data_scope_register(self.goal.domains, self.goal.period)
                if "error" in tool_out:                       # tool results are checked
                    self._log(planned="read the data-scope register before planning",
                              action="tool call: read_data_scope_register()",
                              observed=tool_out, stopped=True,
                              stop_reason=f"tool error: {tool_out['error']}")
                    raise RuntimeError(f"tool error: {tool_out['error']}")
                self.register = tool_out["register"]
                self.conflicts = detect_conflicts(self.goal, self.register, self.goal.period)
                self.goal.conflicts = self.conflicts
                self.goal.scope_checked = True
                self._log(planned="read the data-scope register before locking the plan",
                          action=("tool call: read_data_scope_register("
                                  f"domains={self.goal.domains}, period='{self.goal.period}')"),
                          observed={"access": {d: r["read_access"] for d, r in self.register.items()},
                                    "figure_status": {d: r["figure_status"] for d, r in self.register.items()},
                                    "conflicts_found": len(self.conflicts)})
                for c in self.conflicts:
                    self._log(planned="reconcile the request against observed data scope",
                              action=f"conflict detected: {c['type']} ({c['domain']})",
                              observed={"detail": c["detail"], "required_action": c["required_action"]})

            # ---------- PLAN ----------------------------------------------
            elif step == "generate_plan":
                self.plan = propose_plan(self.goal, self.register, self.conflicts)
                self.goal.plan_locked = True
                self._log(planned="draft the execution plan from the locked requirements",
                          action=(f"built plan: {len(self.plan.body['analysis_steps'])} steps, "
                                  f"{len(self.plan.body['kpi_calculations'])} KPI calculations, "
                                  f"{len(self.conflicts)} caveat(s) carried into assumptions"),
                          observed={"objective": self.plan.body["objective"],
                                    "step_ids": [s["id"] for s in self.plan.body["analysis_steps"]]},
                          stopped=True,
                          stop_reason="goal.plan_locked is now True")
        return self.plan

    # ---- Revise (outside the run loop, on the SAME plan object) ---------
    def apply_feedback(self, feedback: str) -> Plan:
        if self.plan is None:
            raise RuntimeError("no plan to revise")
        before_id, before_rev = id(self.plan), self.plan.revision
        self._log(planned="apply user feedback to the existing plan",
                  action=f"feedback received: {feedback.strip()!r}", observed=None)
        self.plan = revise(self.plan, feedback)
        new_ops = [p for p in self.plan.patch_history if p["revision"] == self.plan.revision]
        self._log(planned="confirm the revision touched only what was asked",
                  action=("mutated the existing plan object in place "
                          "(not regenerated from scratch)"),
                  observed={"same_object": id(self.plan) == before_id,
                            "revision": f"r{before_rev} -> r{self.plan.revision}",
                            "operations": [{"op": p["op"], "why": p["why"], "effect": p["effect"]}
                                           for p in new_ops]},
                  stopped=True,
                  stop_reason="revision applied without restarting the loop")
        return self.plan


def scripted_user(replies: List[str]):
    """Deterministic stand-in for a human at the keyboard, so the run reproduces."""
    box = list(replies)

    def provider(questions, round_no):
        print(f"\n  [coordinator -> user, round {round_no}] {len(questions)} question(s):")
        for i, q in enumerate(questions, 1):
            print(f"     Q{i}. {q}")
        if not box:
            return None
        reply = box.pop(0)
        print(f"  [user -> coordinator] {reply.strip()[:300]}")
        return reply

    return provider


print("CoordinatorAgent ready.")


CoordinatorAgent ready.


### Cell 9 · Run A — the vague request, end to end

`"Analyze our college performance."` — all six requirements unspecified. Watch the agent ask, take the answers, call the tool, observe that Finance 2025-26 is unreconciled, and only then lock a plan that carries that caveat.

In [9]:
# =============================================================================
# Cell 9  ·  RUN A — the vague request from the lab brief, end to end
# =============================================================================
# "Analyze our college performance." — six unspecified requirements. Watch the
# agent ask, take the answers, act on a tool, observe a data conflict, and only
# then lock a plan.

REQUEST_A = "Analyze our college performance."

REPLIES_A = [
    """Cover all four domains - Academic, Research, Placement and Finance - for
       academic year 2025-26. Leadership wants a decision brief with options and risks.""",
    """Use the approved institution-wide KPI set. Compare against NIRF, NAAC and the
       strategic plan. Department-level drill-down only for KPIs that miss their target.""",
]

print("=" * 78)
print("RUN A · EXECUTIVE DECISION SUPPORT SYSTEM · LAB 1 COORDINATOR")
print("=" * 78)
print(f"\nStopping condition (declared before the loop runs): goal.plan_locked is True")
print(f"\nUSER REQUEST: {REQUEST_A!r}")

agent_a = CoordinatorAgent(scripted_user(REPLIES_A))
plan_a = agent_a.run(REQUEST_A)

print("\n" + "-" * 78)
print("REQUIREMENTS AFTER CLARIFICATION")
print("-" * 78)
for slot in REQUIRED_SLOTS:
    print(f"  {slot:<10}: {getattr(agent_a.goal, slot)}")

print("\n" + "-" * 78)
print(f"DATA-SCOPE CONFLICTS SURFACED BY THE TOOL CALL: {len(agent_a.conflicts)}")
print("-" * 78)
for c in agent_a.conflicts:
    print(f"  [{c['type']}] {c['domain']}\n      {c['detail']}\n      -> {c['required_action']}")

print("\n" + "-" * 78)
print("INITIAL PLAN (revision 0)")
print("-" * 78)
print(plan_a.render())


RUN A · EXECUTIVE DECISION SUPPORT SYSTEM · LAB 1 COORDINATOR

Stopping condition (declared before the loop runs): goal.plan_locked is True

USER REQUEST: 'Analyze our college performance.'

  [coordinator -> user, round 1] 6 question(s):
     Q1. Which performance domains must the brief cover — Academic, Research, Placement, Finance, or a subset?
     Q2. Which reporting period does this cover (e.g. academic year 2025-26, or a specific quarter)?
     Q3. Which KPIs should be reported, or should the approved institution-wide KPI set be used in full?
     Q4. Which target set should performance be compared against — NIRF, NAAC, the internal strategic plan, or all three?
     Q5. Is department-level drill-down required for every KPI, or only for KPIs that miss their target?
     Q6. What should leadership receive — a decision brief with options and risks, a dashboard, or both?
  [user -> coordinator] Cover all four domains - Academic, Research, Placement and Finance - for
       academic

### Cell 10 · Assignment 1 — two independent rounds of feedback

> *Extend the reference agent to accept a second, independent round of feedback after the first revision. Both revisions must mutate the same plan object — the log must show two separate revise iterations, neither of which regenerates the plan from scratch.*

The proof is printed, not claimed: `id(plan)` is identical across both rounds, and a field-level diff shows exactly which keys each round touched and which it left alone.

In [10]:
# =============================================================================
# Cell 10 · TWO independent rounds of feedback on the SAME plan object
#           (Lab 1, Assignment 1)
# =============================================================================
# Assignment 1 requires two separate revise iterations, "both of which mutate
# the same plan object", with neither regenerating from scratch. The proof is
# printed below: id(plan) is identical across both rounds, and a field-level
# diff shows exactly which keys the feedback touched and which were untouched.

import copy

PLAN_ID_AT_CREATION = id(plan_a)


def diff_plan(before: Dict[str, Any], after: Dict[str, Any]) -> None:
    changed = [k for k in after if k in before and before[k] != after[k]]
    added = [k for k in after if k not in before]
    untouched = [k for k in after if k in before and before[k] == after[k]]
    print(f"    changed  : {changed or '[]'}")
    print(f"    added    : {added or '[]'}")
    print(f"    untouched: {untouched}")


# ------------------------------ ROUND 1 -------------------------------------
FEEDBACK_1 = """Remove the Finance domain from the analysis.
Make department-level drill-down mandatory for every KPI."""

print("=" * 78)
print("REVISION ROUND 1")
print("=" * 78)
print(f"FEEDBACK:\n{textwrap.indent(FEEDBACK_1, '  ')}")

snapshot_0 = copy.deepcopy(plan_a.body)
plan_a = agent_a.apply_feedback(FEEDBACK_1)

print(f"\n  same object as revision 0 : {id(plan_a) == PLAN_ID_AT_CREATION}"
      f"   (id {PLAN_ID_AT_CREATION})")
print(f"  revision                  : r0 -> r{plan_a.revision}")
print("  operations applied:")
for p in [x for x in plan_a.patch_history if x["revision"] == 1]:
    print(f"    - {p['op']:<14} why: {p['why']}")
    print(f"      effect: {p['effect']}")
print("  field-level diff:")
diff_plan(snapshot_0, plan_a.body)

# ------------------------------ ROUND 2 -------------------------------------
FEEDBACK_2 = """Keep the main brief to two pages.
Every option must name an accountable risk owner.
Also add year-on-year trend context so we can see the direction of travel."""

print("\n" + "=" * 78)
print("REVISION ROUND 2  (independent of round 1, on the same object)")
print("=" * 78)
print(f"FEEDBACK:\n{textwrap.indent(FEEDBACK_2, '  ')}")

snapshot_1 = copy.deepcopy(plan_a.body)
plan_a = agent_a.apply_feedback(FEEDBACK_2)

print(f"\n  same object as revision 0 : {id(plan_a) == PLAN_ID_AT_CREATION}")
print(f"  revision                  : r1 -> r{plan_a.revision}")
print("  operations applied:")
for p in [x for x in plan_a.patch_history if x["revision"] == 2]:
    print(f"    - {p['op']:<14} why: {p['why']}")
    print(f"      effect: {p['effect']}")
print("  field-level diff:")
diff_plan(snapshot_1, plan_a.body)

print("\n" + "-" * 78)
print("FINAL PLAN (revision 2)")
print("-" * 78)
print(plan_a.render())


REVISION ROUND 1
FEEDBACK:
  Remove the Finance domain from the analysis.
  Make department-level drill-down mandatory for every KPI.

  same object as revision 0 : True   (id 4523566400)
  revision                  : r0 -> r1
  operations applied:
    - remove_domain  why: feedback asks to remove the Finance domain
      effect: removed domain Finance: dropped steps ['S5'], its data source and its KPI rows
    - set_field      why: feedback makes department-level drill-down mandatory for every KPI
      effect: set drilldown_policy := 'mandatory for every KPI — each KPI must carry its full department-level trace, not only the KPIs that miss target' (was 'only for KPIs that miss their target')
  field-level diff:
    changed  : ['scope', 'data_required', 'analysis_steps', 'kpi_calculations', 'drilldown_policy', 'assumptions']
    added    : ['_unchanged_note']
    untouched: ['objective', 'target_comparison', 'evidence_requirements', 'brief_structure', 'validation_checks']

REVISION RO

### Cell 11 · Assignment 2 — detect a conflict instead of absorbing it

> *Modify the plan step so that if the request does not actually match what the tool returns, the agent flags the conflict explicitly to the user before finalizing, rather than silently proceeding.*

Two conflict classes are demonstrated across the two runs:

| Conflict | Where | What the agent does |
|---|---|---|
| `PERIOD_NOT_COVERED` + `FIGURES_NOT_FINAL` | Run A — Finance 2025-26 is unreconciled | Flags it, carries it into the plan's assumptions with a required action, and does **not** silently plan over provisional figures |
| `KPI_NOT_SOURCED` | Run B — "carbon footprint" has no linked source | Flags it, plans the rest, and returns the scope decision to leadership rather than inventing a source or quietly dropping the KPI |

Run B doubles as proof that the loop is **not a fixed questionnaire**: the same agent, given a request that already answers all six slots, asks zero questions.

In [11]:
# =============================================================================
# Cell 11 · RUN B — a fully-specified request, and a second class of conflict
#           (Lab 1, Assignment 2)
# =============================================================================
# Two things are demonstrated here:
#
#  1. NOT A FIXED QUESTIONNAIRE. The same agent, given a request that already
#     answers all six slots, asks zero questions and goes straight to acting.
#     The number of questions is a function of goal state, not of the script.
#
#  2. A SECOND CONFLICT TYPE. Assignment 2 asks for a conflict the agent must
#     flag rather than silently absorb. Run A surfaced provisional Finance
#     figures; Run B asks for "carbon footprint", which no linked source system
#     reports. The agent refuses to plan around a metric it cannot source.

REQUEST_B = """Prepare a two-page decision brief on Placement and Research performance for
academic year 2025-26, benchmarked against NIRF and our strategic plan. Report placement
percentage, average package, publications, citations and carbon footprint, with
department-level drill-down for every KPI."""

print("=" * 78)
print("RUN B · A REQUEST THAT ALREADY SPECIFIES EVERYTHING")
print("=" * 78)
print(textwrap.indent(textwrap.fill(REQUEST_B, 74), "  "))

agent_b = CoordinatorAgent(scripted_user([]))     # no replies staged — none needed
plan_b = agent_b.run(REQUEST_B)

clarify_rounds = [e for e in agent_b.log if e["planned"].startswith("clarify")]
questions_asked = sum(len(e["observed"]["questions"]) for e in clarify_rounds)

print(f"\n  clarification rounds : {len(clarify_rounds)}")
print(f"  questions asked      : {questions_asked}   <- 0, because nothing was missing")
print("  slots extracted from the request text alone:")
for slot in REQUIRED_SLOTS:
    print(f"      {slot:<10}: {getattr(agent_b.goal, slot)}")

print(f"\n  conflicts surfaced   : {len(agent_b.conflicts)}")
for c in agent_b.conflicts:
    print(f"      [{c['type']}] {c['domain']}")
    print(f"          {c['detail']}")
    print(f"          -> {c['required_action']}")

unsourced = [c for c in agent_b.conflicts if c["type"] == "KPI_NOT_SOURCED"]
print("\n  ASSIGNMENT 2 CHECK — unsourceable KPI is reported, not silently dropped:")
print(f"      flagged            : {bool(unsourced)}")
print(f"      carried into plan  : "
      f"{any('KPI_NOT_SOURCED' in a for a in plan_b.body['assumptions'])}")
print("      the agent did NOT invent a source, and did NOT quietly remove the KPI —")
print("      it planned the rest and put the decision back to leadership.")

print("\n" + "-" * 78)
print("RUN B PLAN — scope and assumptions")
print("-" * 78)
print(f"  domains : {plan_b.body['scope']['domains']}")
print(f"  steps   : {[s['id'] for s in plan_b.body['analysis_steps']]}")
print("  assumptions:")
for a in plan_b.body["assumptions"]:
    print(textwrap.fill(f"    - {a}", 88, subsequent_indent="      "))


RUN B · A REQUEST THAT ALREADY SPECIFIES EVERYTHING
  Prepare a two-page decision brief on Placement and Research performance
  for academic year 2025-26, benchmarked against NIRF and our strategic
  plan. Report placement percentage, average package, publications,
  citations and carbon footprint, with department-level drill-down for every
  KPI.

  clarification rounds : 1
  questions asked      : 0   <- 0, because nothing was missing
  slots extracted from the request text alone:
      domains   : ['Research', 'Placement']
      period    : academic year 2025-26
      kpis      : ['publications', 'citations', 'placement percentage', 'average package', 'carbon footprint']
      targets   : ['NIRF', 'strategic plan']
      drilldown : mandatory for every KPI
      output    : decision brief with options and risks

  conflicts surfaced   : 2
      [SCOPE_CAVEAT] Placement
          3 provisional offers are excluded until the Placement Cell verifies them.
          -> Record the caveat 

### Cell 12 · Reflection — chatbot-like vs agent-like

> *Students should identify which steps are chatbot-like and which are agent-like.* — instructor prompt

The same vague request is put through a one-shot chatbot first, then compared against the agent using numbers taken from the actual run logs.

In [12]:
# =============================================================================
# Cell 12 · Reflection: which steps are chatbot-like, which are agent-like
# =============================================================================
# The instructor prompt asks students to identify this explicitly. So here is
# the same vague request put through a one-shot chatbot, then a measured
# comparison built from the actual run logs above — not from claims.

CHATBOT_SYSTEM = "You are a helpful assistant."
CHATBOT_SCHEMA = {"type": "object",
                  "properties": {"answer": {"type": "string"}},
                  "required": ["answer"]}


@llm.offline.register("chatbot")
def _offline_chatbot(payload):
    return {"answer": (
        "Here is an analysis of your college performance for the current year. Academic "
        "performance is strong, research output is growing steadily, placements are "
        "healthy and the budget is well utilised. I recommend focusing on research "
        "output and improving placement outcomes in the weaker departments.")}


one_shot = llm.json(task="chatbot", system=CHATBOT_SYSTEM,
                    prompt=f"Answer this request directly:\n{REQUEST_A}",
                    schema=CHATBOT_SCHEMA, payload={})

print("=" * 78)
print("SAME REQUEST, ONE-SHOT CHATBOT  (the failure mode this lab is about)")
print("=" * 78)
print(textwrap.indent(textwrap.fill(one_shot["answer"], 74), "  "))
print("""
  What went wrong, concretely:
    - It asked nothing, so 'our college', 'performance' and the period stayed undefined.
    - It called no tool, so it never learned that Finance 2025-26 is unreconciled.
    - It asserted a conclusion ("budget is well utilised") that a tool should verify.
    - There is nothing to revise: the output is prose, so any change means regenerating.
""")

# ---- Measured comparison, computed from the real logs -----------------------
tool_calls_a = [e for e in agent_a.log if "tool call:" in e["action"]]
clarify_a = [e for e in agent_a.log if e["planned"].startswith("clarify")]
questions_a = sum(len(e["observed"]["questions"]) for e in clarify_a)
revisions_a = len({p["revision"] for p in plan_a.patch_history})

rows = [
    ("Clarifying questions asked",      "0",  str(questions_a)),
    ("Tool calls before committing",    "0",  str(len(tool_calls_a))),
    ("Loop iterations logged",          "1",  str(len(agent_a.log))),
    ("Data conflicts caught",           "0",  str(len(agent_a.conflicts))),
    ("Revision rounds on one object",   "0",  str(revisions_a)),
    ("Stopping condition declared",     "no", "yes (goal.plan_locked)"),
    ("Output is machine-editable",      "no", "yes (typed Plan object)"),
]
print("=" * 78)
print(f"{'BEHAVIOUR':<34}{'CHATBOT':<12}{'AGENT (this notebook)'}")
print("=" * 78)
for label, cb, ag in rows:
    print(f"{label:<34}{cb:<12}{ag}")

print("""
Mapping to the Goal -> Plan -> Act -> Observe -> Revise loop:

  Goal     Cell 3   goal slots + `plan_locked` as the stopping predicate
  Plan     Cell 8   next_step() picks the next action from goal state alone
  Act      Cell 5   read_data_scope_register() — a real call with a checked return
  Observe  Cell 8   the register's answer becomes conflicts folded into the plan
  Revise   Cell 7   typed patch ops mutate the live Plan; the loop never restarts

  Chatbot-like steps in this notebook : the wording of questions and of plan prose.
  Agent-like steps                    : deciding WHICH question to ask, whether to
                                        call the tool, when to stop, and what to
                                        change on feedback — all driven by state.
""")


SAME REQUEST, ONE-SHOT CHATBOT  (the failure mode this lab is about)
  Here is an analysis of your college performance for the current year.
  Academic performance is strong, research output is growing steadily,
  placements are healthy and the budget is well utilised. I recommend
  focusing on research output and improving placement outcomes in the weaker
  departments.

  What went wrong, concretely:
    - It asked nothing, so 'our college', 'performance' and the period stayed undefined.
    - It called no tool, so it never learned that Finance 2025-26 is unreconciled.
    - It asserted a conclusion ("budget is well utilised") that a tool should verify.
    - There is nothing to revise: the output is prose, so any change means regenerating.

BEHAVIOUR                         CHATBOT     AGENT (this notebook)
Clarifying questions asked        0           9
Tool calls before committing      0           1
Loop iterations logged            1           15
Data conflicts caught            

### Cell 13 · Acceptance checks — the rubric, executed

Sixteen checks covering the Worklet 15 Lab 1 acceptance line, the lab-guide rubric, and both assignments. Each one either passes or raises — nothing here is self-reported.

In [13]:
# =============================================================================
# Cell 13 · Acceptance checks — the rubric, executed
# =============================================================================
# Worklet 15, Lab 1 acceptance: "Given a vague request it asks at least two
# clarifying questions and yields a coherent, editable plan."
# Lab guide rubric: goal as data, stopping condition declared up front, ask only
# what is missing, a real tool call with a checked return, and revise() that
# mutates rather than regenerates.
#
# Each check below either passes or raises. Nothing here is self-reported.

checks: List[tuple] = []


def check(name, condition, evidence=""):
    checks.append((name, bool(condition), evidence))
    return bool(condition)


# 1 — at least two clarifying questions on the vague request
first_round = [e for e in agent_a.log if e["planned"].startswith("clarify")][0]
q1 = len(first_round["observed"]["questions"])
check("A1  vague request triggers >= 2 clarifying questions", q1 >= 2,
      f"{q1} questions asked in round 1")

# 2 — the agent asks only about slots that are actually empty
b_rounds = [e for e in agent_b.log if e["planned"].startswith("clarify")]
b_questions = sum(len(e["observed"]["questions"]) for e in b_rounds)
check("A2  fully-specified request triggers 0 questions (not a questionnaire)",
      b_questions == 0, f"Run B asked {b_questions} questions")

# 3 — round 2 of Run A re-asked nothing that round 1 had already answered
r1_q = set(agent_a.log[1]["observed"]["questions"])
r2 = [e for e in agent_a.log if e["planned"].startswith("clarify")][1]
r2_q = set(r2["observed"]["questions"])
check("A3  no question is ever repeated once answered", r2_q < r1_q,
      f"round 1 asked {len(r1_q)}, round 2 asked {len(r2_q)} (a strict subset)")

# 4 — a real tool call happened, and it happened BEFORE the plan was locked
tool_idx = next(i for i, e in enumerate(agent_a.log) if "tool call:" in e["action"])
lock_idx = next(i for i, e in enumerate(agent_a.log)
                if e.get("stop_reason") == "goal.plan_locked is now True")
check("A4  Act/Observe: tool called, and called before the plan was locked",
      tool_idx < lock_idx, f"tool at iteration {tool_idx + 1}, plan locked at {lock_idx + 1}")

# 5 — the tool's return value was actually checked, not assumed
check("A5  tool contract honoured (error branch exists and is exercised)",
      "error" in read_data_scope_register(["Sustainability"], "2025-26"),
      "read_data_scope_register(['Sustainability']) -> {'error': ...}")

# 6 — the plan carries every required section
REQUIRED_PLAN_KEYS = ["objective", "scope", "data_required", "analysis_steps",
                      "kpi_calculations", "target_comparison", "drilldown_policy",
                      "evidence_requirements", "brief_structure", "validation_checks",
                      "assumptions"]
missing_keys = [k for k in REQUIRED_PLAN_KEYS if k not in plan_a.body]
check("A6  plan contains all 11 required sections", not missing_keys,
      f"missing: {missing_keys or 'none'}")

# 7 — the plan is EDITABLE: every analysis step is individually addressable
ids = [s["id"] for s in plan_a.body["analysis_steps"]]
check("A7  plan is editable (every step has a unique id)",
      len(ids) == len(set(ids)) and all(ids), f"step ids: {ids}")

# 8 — two revision rounds, both on the same object (Assignment 1)
check("A8  two revision rounds applied", plan_a.revision == 2,
      f"plan.revision == {plan_a.revision}")
check("A9  both revisions mutated the SAME object (no regeneration)",
      id(plan_a) == PLAN_ID_AT_CREATION,
      f"id at creation {PLAN_ID_AT_CREATION} == id now {id(plan_a)}")
check("A10 each revision is individually auditable",
      {p["revision"] for p in plan_a.patch_history} == {1, 2}
      and all(p.get("why") for p in plan_a.patch_history),
      f"{len(plan_a.patch_history)} recorded operations, each with a 'why'")

# 9 — revision 2 provably left untouched everything the feedback did not mention
check("A11 revision 2 left the objective and KPI formulas untouched",
      plan_a.body["objective"] == snapshot_1["objective"]
      and plan_a.body["kpi_calculations"] == snapshot_1["kpi_calculations"],
      "byte-identical before and after round 2")

# 10 — conflicts observed from the tool survived into the plan (Assignment 2)
check("A12 every observed data conflict is carried into the plan",
      all(any(c["type"] in a for a in plan_a.body["assumptions"])
          for c in agent_a.conflicts),
      f"{len(agent_a.conflicts)} conflicts, all present in assumptions")

# 11 — the agent invented no institutional figures
figure_like = re.findall(r"\d+(?:\.\d+)?\s?%", json.dumps(plan_a.body))
check("A13 no institutional figure was fabricated (Lab 1 produces a plan only)",
      not figure_like, f"numeric percentage claims found: {figure_like or 'none'}")

# 12 — every log entry is a complete iteration record
complete = all({"planned", "action", "observed", "stopped"} <= set(e)
               for e in agent_a.log)
check("A14 iteration log records planned/action/observed/stopped for every step",
      complete, f"{len(agent_a.log)} log entries")

# 13 — the loop stopped for the reason declared before it ran
check("A15 the loop stopped for its declared stopping condition",
      agent_a.log[lock_idx]["stop_reason"] == "goal.plan_locked is now True",
      agent_a.log[lock_idx]["stop_reason"])

# 14 — the agent refuses to proceed on incomplete requirements
stubborn = CoordinatorAgent(scripted_user(["I'm not sure, just do whatever is standard."] * 5))
stubborn_plan = stubborn.run("Give me a performance report.")
check("A16 refuses to invent missing requirements when the user never supplies them",
      stubborn_plan is None and not stubborn.goal.plan_locked,
      stubborn.log[-1].get("stop_reason", ""))

print("=" * 78)
print("ACCEPTANCE CHECKS")
print("=" * 78)
for name, ok, ev in checks:
    print(f"  [{'PASS' if ok else 'FAIL'}]  {name}")
    if ev:
        print(f"          {ev}")
failed = [n for n, ok, _ in checks if not ok]
print("=" * 78)
print(f"  {len(checks) - len(failed)}/{len(checks)} checks passed.")
assert not failed, f"FAILED CHECKS: {failed}"
print("  Lab 1 acceptance criteria satisfied.")



  [coordinator -> user, round 1] 6 question(s):
     Q1. Which performance domains must the brief cover — Academic, Research, Placement, Finance, or a subset?
     Q2. Which reporting period does this cover (e.g. academic year 2025-26, or a specific quarter)?
     Q3. Which KPIs should be reported, or should the approved institution-wide KPI set be used in full?
     Q4. Which target set should performance be compared against — NIRF, NAAC, the internal strategic plan, or all three?
     Q5. Is department-level drill-down required for every KPI, or only for KPIs that miss their target?
     Q6. What should leadership receive — a decision brief with options and risks, a dashboard, or both?
  [user -> coordinator] I'm not sure, just do whatever is standard.

  [coordinator -> user, round 2] 6 question(s):
     Q1. Which performance domains must the brief cover — Academic, Research, Placement, Finance, or a subset?
     Q2. Which reporting period does this cover (e.g. academic year 2025-2

### Cell 14 · Deliverables

The Lab 1 deliverable is *"prompt log, plan, revised plan, and reflection on agent behavior"*. All four are produced above; the logs and plans are also exported as JSON for attaching to the submission.

In [14]:
# =============================================================================
# Cell 14 · Deliverables — full iteration log, prompt log, and exported artifacts
# =============================================================================
# The Lab 1 deliverable is "prompt log, plan, revised plan, and reflection on
# agent behavior". All four are produced here; the first three are also written
# to disk so they can be attached to the submission.

print("=" * 78)
print("FULL ITERATION LOG — RUN A")
print("=" * 78)
for e in agent_a.log:
    print(f"\n[{e['iteration']:>2}] planned : {e['planned']}")
    print(f"     action  : {e['action']}")
    obs = e["observed"]
    obs_text = json.dumps(obs, indent=2, ensure_ascii=False) if not isinstance(obs, str) else obs
    print("     observed: " + textwrap.indent(obs_text, " " * 15).lstrip())
    print(f"     state   : missing={e['goal_state']['missing']}  "
          f"scope_checked={e['goal_state']['scope_checked']}  "
          f"plan_locked={e['goal_state']['plan_locked']}")
    print(f"     stopped : {e['stopped']}"
          + (f"   reason: {e['stop_reason']}" if e.get("stop_reason") else ""))

SHOW_CALLS = 8
print("\n" + "=" * 78)
print(f"PROMPT LOG — {len(llm.calls)} model call(s), backend '{llm.mode}'"
      f"   (first {SHOW_CALLS} shown; all of them are in the JSON export below)")
print("=" * 78)
for c in llm.calls[:SHOW_CALLS]:
    print(f"\n--- call {c['call']}  task={c['task']}  backend={c['backend']} ---")
    print(textwrap.indent(textwrap.shorten(c["prompt"].replace("\n", " "), 420), "  PROMPT   : "))
    print(textwrap.indent(textwrap.shorten(json.dumps(c["response"], ensure_ascii=False), 420),
                          "  RESPONSE : "))

# ---- Export -----------------------------------------------------------------
from pathlib import Path
outdir = Path("lab1_artifacts")
outdir.mkdir(exist_ok=True)

artifacts = {
    "lab1_iteration_log.json":  agent_a.log,
    "lab1_prompt_log.json":     llm.calls,
    "lab1_plan_final.json":     plan_a.to_dict(),
    "lab1_run_b_plan.json":     plan_b.to_dict(),
    "lab1_acceptance_checks.json": [{"check": n, "passed": ok, "evidence": ev}
                                    for n, ok, ev in checks],
}
for fname, obj in artifacts.items():
    (outdir / fname).write_text(json.dumps(obj, indent=2, ensure_ascii=False, default=str))

print("\n" + "=" * 78)
print("ARTIFACTS WRITTEN")
print("=" * 78)
for fname in artifacts:
    print(f"  {outdir / fname}  ({(outdir / fname).stat().st_size:,} bytes)")
print("\n(In Colab: open the file browser on the left, or use "
      "`from google.colab import files; files.download(...)`.)")


FULL ITERATION LOG — RUN A

[ 1] planned : ingest the leadership request
     action  : received free-text request
     observed: Analyze our college performance.
     state   : missing=['domains', 'period', 'kpis', 'targets', 'drilldown', 'output']  scope_checked=False  plan_locked=False
     stopped : False

[ 2] planned : clarify 6 unspecified requirement(s): ['domains', 'period', 'kpis', 'targets', 'drilldown', 'output']
     action  : round 1: extracted nothing from the request text; asked 6 question(s)
     observed: {
                 "questions": [
                   "Which performance domains must the brief cover — Academic, Research, Placement, Finance, or a subset?",
                   "Which reporting period does this cover (e.g. academic year 2025-26, or a specific quarter)?",
                   "Which KPIs should be reported, or should the approved institution-wide KPI set be used in full?",
                   "Which target set should performance be compared against — NIR

### Cell 15 · Optional — drive the coordinator by hand

Identical agent, identical loop; only the source of the answers changes. Off by default so the notebook executes unattended and reproducibly.

In [15]:
# =============================================================================
# Cell 15 · Optional: drive the same coordinator interactively
# =============================================================================
# Identical agent, identical loop — the only thing that changes is where the
# answers come from. Set INTERACTIVE = True and run this cell to talk to it.
# It is off by default so the notebook executes top-to-bottom unattended.

INTERACTIVE = False


def interactive_user(questions, round_no):
    print(f"\n[round {round_no}] The coordinator needs {len(questions)} thing(s):")
    for i, q in enumerate(questions, 1):
        print(f"   Q{i}. {q}")
    return input("\nYour reply (one message, answer what you can):\n> ")


if INTERACTIVE:
    live = CoordinatorAgent(interactive_user)
    live_plan = live.run(input("Enter your request for the leadership brief:\n> "))
    if live_plan:
        print("\n" + live_plan.render())
        while input("\nRevise the plan? (yes/no): ").strip().lower().startswith("y"):
            live.apply_feedback(input("What should change?\n> "))
            print("\n" + live_plan.render())
        print("\n=== FINAL PLAN ===")
        print(live_plan.render())
else:
    print("INTERACTIVE = False — set it to True and re-run this cell to drive the agent by hand.")
    print("Everything above ran from a scripted transcript so the notebook is reproducible.")


INTERACTIVE = False — set it to True and re-run this cell to drive the agent by hand.
Everything above ran from a scripted transcript so the notebook is reproducible.


---

## Where Lab 1 leaves the worklet

**Nodes activated:** the Coordinator loop. No specialist agents yet — the Academic / Research / Placement / Finance / KPI / Report-Generation agents arrive from Lab 7 onward.

**What exists now:** a coordinator that will not act on an underspecified request, that reads a real source before committing, that carries data caveats into its plan instead of hiding them, and whose plan can be edited surgically and audited afterwards.

**What Lab 2 adds:** the tool layer. Right now the coordinator reads a *scope register* — who may read what, and how final it is. Lab 2 gives it typed tools that read the department reports and the KPI definitions themselves and **compute** the numbers, so no institutional figure is ever asserted in a prompt.

> Lab 2 pseudocode (Chapter 15):
> ```
> a = tool.read('institutional KPIs & dashboards')
> b = tool.read('department reports')
> v = tool.compute(a, b)
> assert v.valid
> ```